In [11]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# =========================
# 1️⃣ 初始化模型
# =========================
model = init_chat_model(
    model="ollama:qwen3.5:4b"
    )

# =========================
# 2️⃣ 定义多个 Agent
# =========================

math_agent = create_agent(model=model,system_prompt="你是一个数学专家，只回答数学计算问题")

code_agent = create_agent(model=model,system_prompt="你是一个Python编程专家")

chat_agent = create_agent(model=model,system_prompt="你是一个日常聊天助手")

# =========================
# 3️⃣ Router（核心）
# =========================
from langchain.messages import HumanMessage
from pydantic import BaseModel,Field
from typing import Literal


class Classification(BaseModel):
    """将用户的查询做分类"""
    
    category: Literal["math", "code", "chat"] = Field(description="用户问题的类别")
    
structured_model = model.with_structured_output(Classification)


def llm_router(query):
    prompt = f"""
你是一个路由器，请判断用户问题属于哪一类：

可选：
1. math
2. code
3. chat

只输出类别，不要解释

用户问题：
{query}
"""
    res = structured_model.invoke([HumanMessage(content=prompt + "\n用户问题：" + query)])
    return res.category




#============================
#4️⃣ 调度器
#============================
def run(query: str):
    route = llm_router(query)

    if route == "math":
        return math_agent.invoke({"messages": query})
    elif route == "code":
        return code_agent.invoke({"messages": query})
    else:
        return chat_agent.invoke({"messages": query})

# ===========================
# 5️⃣ 测试
# ===========================
from rich import print as rprint

rprint(run("1+1等于多少？"))
rprint(run("写一个python冒泡排序"))
rprint(run("你是谁？"))

{
    'messages': [
        HumanMessage(
            content='1+1等于多少？',
            additional_kwargs={},
            response_metadata={},
            id='2adb720c-c4b2-4f56-a55f-785b5e8d6b72'
        ),
        AIMessage(
            content='1 + 1 = 2',
            additional_kwargs={},
            response_metadata={
                'model': 'qwen3.5:4b',
                'created_at': '2026-04-29T11:47:00.112759Z',
                'done': True,
                'done_reason': 'stop',
                'total_duration': 54853398708,
                'load_duration': 120134791,
                'prompt_eval_count': 30,
                'prompt_eval_duration': 272505125,
                'eval_count': 680,
                'eval_duration': 54282119453,
                'logprobs': None,
                'model_name': 'qwen3.5:4b',
                'model_provider': 'ollama'
            },
            id='lc_run--019dd90f-8148-75b2-9fe4-887c2db065ca-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 30, 'output_tokens': 680, 'total_tokens': 710}
        )
    ]
}

{
    'messages': [
        HumanMessage(
            content='写一个python冒泡排序',
            additional_kwargs={},
            response_metadata={},
            id='478ae8d1-82a2-41bd-81da-2e7a48bc0f61'
        ),
        AIMessage(
            content='以下是 Python 
中冒泡排序的两种实现方式：基础版和优化版，包含使用示例及说明。冒泡排序是一种简单直观的排序算法，适用于学习排序原理
和小型数据集处理。\n\n---\n\n## 1. 基础版冒泡排序\n\n```python\ndef bubble_sort_basic(arr):\n    n = len(arr)\n    
# 遍历数组，每一轮将未排序部分的最后一个元素移到正确位置\n    for i in range(n - 1):\n        for j in range(0, n -
1 - i):\n            if arr[j] > arr[j + 1]:\n                arr[j], arr[j + 1] = arr[j + 1], arr[j]\n    return 
arr\n\n# 使用示例\ndata = [64, 34, 25, 12, 22, 11, 90]\nprint("原始数组:", data)\nsorted_data = 
bubble_sort_basic(data.copy())\nprint("排序后数组:", sorted_data)\n```\n\n---\n\n## 2. 
优化版冒泡排序（带提前退出机制）\n\n```python\ndef bubble_sort_optimized(arr):\n    n = len(arr)\n    swapped = 
True\n    while swapped:\n        swapped = False\n        for i in range(n - 1):\n            if arr[i] > arr[i + 
1]:\n                arr[i], arr[i + 1] = arr[i + 1], arr[i]\n                swapped = True\n    return arr\n\n# 
使用示例\ndata = [64, 34, 25, 12, 22, 11, 90]\nprint("原始数组:", data)\nsorted_data = 
bubble_sort_optimized(data.copy())\nprint("排序后数组:", sorted_data)\n```\n\n---\n\n## 3. 
可选：带详细步骤输出的版本（教学用途）\n\n```python\ndef bubble_sort_trace(arr):\n    print(f"初始数组: {arr}")\n  
n = len(arr)\n    for i in range(n - 1):\n        for j in range(n - 1 - i):\n            if arr[j] > arr[j + 1]:\n
arr[j], arr[j + 1] = arr[j + 1], arr[j]\n                print(f"比较并交换: {arr}，第{j + 1}轮第{i}次")\n    
print(f"最终数组: {arr}")\n    return arr\n```\n\n---\n\n## 性能分析与比较\n\n| 实现版本 | 时间复杂度 | 空间复杂度 
| 特点说明           |\n|----------|------------|------------|--------------------|\n| 基础版   | O(n²)      | O(1)
| 简单易懂，适合教学 |\n| 优化版   | O(n²) 最坏<br>O(k) 最好（k为已排序轮数） | O(1) | 可减少比较次数     
|\n\n---\n\n## 注意事项\n\n- 冒泡排序属于原地排序，不需要额外空间。\n- 
对于大型数据集，建议使用更高效的排序算法（如快速排序、归并排序或 Python 内置的 `list.sort()`）。\n- 
实际应用中不建议在大数据量时手动实现冒泡排序，除非用于学习或处理小规模数据。\n\n---\n\n如果你需要可视化动画、统计性
能，或想比较其他排序算法，请告诉我，我可以提供相应代码。',
            additional_kwargs={},
            response_metadata={
                'model': 'qwen3.5:4b',
                'created_at': '2026-04-29T11:52:58.327609Z',
                'done': True,
                'done_reason': 'stop',
                'total_duration': 66140428500,
                'load_duration': 106245333,
                'prompt_eval_count': 25,
                'prompt_eval_duration': 265983125,
                'eval_count': 820,
                'eval_duration': 65550962708,
                'logprobs': None,
                'model_name': 'qwen3.5:4b',
                'model_provider': 'ollama'
            },
            id='lc_run--019dd914-cc77-74d3-99fe-9a72bac3047d-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 25, 'output_tokens': 820, 'total_tokens': 845}
        )
    ]
}

{
    'messages': [
        HumanMessage(
            content='你是谁？',
            additional_kwargs={},
            response_metadata={},
            id='33d54cc7-25bb-4ec5-a3d6-1355703efbf7'
        ),
        AIMessage(
            content='你好呀！😊 
我是你的日常聊天助手，很高兴认识你！\n\n以后不管是想找人聊聊心事、分享日常，还是有其他小问题，都可以随时找我～\n今
天过得怎么样？想跟我聊点什么？',
            additional_kwargs={},
            response_metadata={
                'model': 'qwen3.5:4b',
                'created_at': '2026-04-29T11:54:43.987283Z',
                'done': True,
                'done_reason': 'stop',
                'total_duration': 66578214292,
                'load_duration': 114701167,
                'prompt_eval_count': 21,
                'prompt_eval_duration': 259610334,
                'eval_count': 760,
                'eval_duration': 65983380082,
                'logprobs': None,
                'model_name': 'qwen3.5:4b',
                'model_provider': 'ollama'
            },
            id='lc_run--019dd916-677e-7210-baa3-57a6ad9a3a7b-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={'input_tokens': 21, 'output_tokens': 760, 'total_tokens': 781}
        )
    ]
}